# Huấn Luyện Nhận Diện 20 Gloss ASL-Citizen trên Kaggle

> **LƯU Ý QUAN TRỌNG KHI CHẠY TRÊN KAGGLE:**
> 1. **Bật Internet**: Tại panel bên phải, mục **Settings** -> gạt **Internet: On** (Kaggle yêu cầu tài khoản đã xác minh số điện thoại) để tải video qua HTTP Range.
> 2. **Chọn Accelerator**: Khuyến nghị chọn **GPU T4** hoặc **CPU** (MediaPipe và GRU cho 600 video chạy rất nhẹ nhàng).

### Pipeline hoàn chỉnh trong Notebook:
1. **Cài đặt môi trường & nạp cấu hình** 20 gloss mục tiêu.
2. **Tải đúng 619 video của 20 gloss** (~360 MB) qua giao thức HTTP Range từ `ASL_Citizen.zip` (nhanh, chỉ mất ~1-2 phút).
3. **Trích xuất Hand Landmarks với MediaPipe Hands**: Sắp xếp tay trội (Dominant Hand), dời cổ tay, palm scale, tọa độ toàn cục, vận tốc và mask (143 chiều/frame, T=32).
4. **Kiểm tra tính lành mạnh (Sanity Check)**: Huấn luyện overfit trên 20 video (loss tiệm cận 0, acc 100%) để đảm bảo không có bug.
5. **Baseline Random Forest**: Đánh giá chuẩn ngẫu nhiên trên đặc trưng thống kê thời gian.
6. **Huấn luyện mô hình PyTorch GRU**: Chạy với 3 seeds, Early Stopping (patience 15), Label Smoothing 0.1, Weight Decay 1e-4.
7. **Đánh giá trên Test Set**: Báo cáo Top-1, Top-3, Macro-F1, Confusion Matrix và phân tích theo từng Signer.

## 1. Setup Thư Viện & Môi Trường

In [ ]:
import os, sys, time, json, shutil, subprocess, importlib
from pathlib import Path

# 1. Cài đặt các gói phụ trợ nếu chưa có
for pkg in ["mediapipe", "remotezip", "pyyaml", "tqdm", "scikit-learn"]:
    try:
        importlib.import_module(pkg.replace("-", "_"))
    except ImportError:
        print(f"Cài đặt {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg], stdout=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# 2. Xác định đường dẫn repo hoặc tải repo từ GitHub nếu chạy độc lập
repo_root = Path.cwd().resolve()
for p in [repo_root, *repo_root.parents]:
    if (p / "configs" / "config.yaml").exists():
        repo_root = p
        break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Thiết bị tính toán
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Thư mục dự án: {repo_root}")
print(f" Thiết bị tính toán: {device}")

## 2. Nạp Cấu Hình & Danh Sách 20 Gloss Mục Tiêu

In [ ]:
from src.data.asl_citizen import resolve_paths
from src.data.preprocessing import create_splits_from_csv, select_glosses, verify_splits

P = resolve_paths(repo_root)
splits_dir = P.splits

# Đọc hoặc sinh lại các file split
meta_path = P.metrics / "full_eda_bundle" / "meta_full.csv"
if not meta_path.exists():
    meta_path = repo_root / "outputs" / "metrics" / "full_eda_bundle" / "meta_full.csv"

if (splits_dir / "classes.csv").exists():
    classes_df = pd.read_csv(splits_dir / "classes.csv")
    glosses = classes_df["gloss"].tolist()
else:
    glosses = select_glosses(k=20)
    if meta_path.exists():
        create_splits_from_csv(meta_path, glosses, splits_dir)

print("=" * 60)
print(f"20 GLOSS MỤC TIÊU ({len(glosses)} classes):")
print(glosses)
print("=" * 60)

# Kiểm tra tính toàn vẹn của split
summary = verify_splits(splits_dir)
print(f" - Tập Train: {summary['train']['videos']} video ({summary['train']['signers']} signers)")
print(f" - Tập Val:   {summary['val']['videos']} video ({summary['val']['signers']} signers)")
print(f" - Tập Test:  {summary['test']['videos']} video ({summary['test']['signers']} signers)")

## 3. Tải Toàn Bộ Video Cho 20 Gloss (HTTP Range Download)

Tải trực tiếp đúng 619 video (~360 MB) vào thư mục `videos/` qua HTTP Range Zip, bỏ qua các video đã có sẵn.

In [ ]:
from src.data.asl_citizen import DEFAULT_CFG, download_videos

P.videos.mkdir(parents=True, exist_ok=True)

# Tập hợp danh sách video cần tải
target_videos = set()
for s in ["train", "val", "test"]:
    sp_df = pd.read_csv(splits_dir / f"{s}.csv")
    target_videos.update(sp_df["video_file"].tolist())

print(f"Số video cần tải: {len(target_videos)} video")

# Đọc kích thước file từ meta_full.csv hoặc zip index
if meta_path.exists():
    full_meta = pd.read_csv(meta_path)
    pool_df = full_meta[full_meta["video_file"].isin(target_videos)][["video_file", "zip_name", "file_size"]].drop_duplicates()
else:
    from src.data.asl_citizen import open_zip, index_zip
    zf = open_zip(DEFAULT_CFG["zip_src"])
    idx = index_zip(zf)
    idx_map = {Path(r["name"]).name: r for r in idx.to_dict("records")}
    pool_rows = [{"video_file": v, "zip_name": idx_map[v]["name"], "file_size": int(idx_map[v]["file_size"])} for v in target_videos if v in idx_map]
    pool_df = pd.DataFrame(pool_rows)

print(f"Dung lượng nén: {pool_df['file_size'].sum() / 1e6:.2f} MB")
download_log = download_videos(
    src=DEFAULT_CFG["zip_src"],
    pool_df=pool_df,
    out_dir=P.videos,
    workers=8,
    log_path=P.metrics / "kaggle_download_log.csv"
)

downloaded = sum(1 for v in target_videos if (P.videos / v).exists() and (P.videos / v).stat().st_size > 0)
print(f" 100% Sẵn sàng: {downloaded}/{len(target_videos)} video trên đĩa.")

## 4. Trích Xuất Hand Landmarks & Chuẩn Hóa Đặc Trưng (143 chiều)

Sử dụng `MediaPipeHandExtractor` kết hợp sắp xếp tay trội (Dominant Hand), dời cổ tay, palm scale, tọa độ toàn cục, vận tốc và mask. Kết quả lưu ra các file `.npy`.

In [ ]:
from src.features.landmark_extractor import MediaPipeHandExtractor
from src.features.normalization import process_video_landmarks

landmarks_dir = repo_root / "data" / "landmarks"
landmarks_dir.mkdir(parents=True, exist_ok=True)

extractor = MediaPipeHandExtractor()
all_samples = pd.concat([pd.read_csv(splits_dir / f"{s}.csv") for s in ["train", "val", "test"]])
gloss_rates = {}

print(f"Đang trích xuất landmarks cho {len(all_samples)} video...")
for _, row in tqdm(all_samples.iterrows(), total=len(all_samples)):
    vid_id = row["video_id"]
    vfile = row["video_file"]
    gloss = row["gloss"]
    out_npy = landmarks_dir / f"{vid_id}.npy"

    if out_npy.exists() and out_npy.stat().st_size > 0:
        continue

    vpath = P.videos / vfile
    if not vpath.exists():
        continue

    raw_coords, hand_masks, stats = extractor.extract_from_video(vpath)
    if stats.get("ok", False) and len(raw_coords) > 0:
        gloss_rates.setdefault(gloss, []).append(stats.get("det_rate", 0.0))
        feat_143, _ = process_video_landmarks(raw_coords, hand_masks, target_len=32)
        np.save(str(out_npy), feat_143.astype(np.float32))

extractor.close()
print(f" Trích xuất hoàn tất! Số file .npy hiện có: {len(list(landmarks_dir.glob('*.npy')))}")

## 5. Sanity Check: Overfit trên 20 Video (Tắt Dropout, Tắt Augmentation)

Xác nhận pipeline lành mạnh: huấn luyện 1 video cho mỗi class trong 40 epochs. Kỳ vọng Loss -> 0 và Accuracy -> 100%.

In [ ]:
from src.data.dataset import ASLLandmarkDataset
from src.models.gru import ASLGRUClassifier
from torch.utils.data import DataLoader, Subset

train_dataset = ASLLandmarkDataset(splits_dir / "train.csv", landmarks_dir, is_train=True, augment=False)

# Chọn đúng 1 mẫu cho mỗi class
mini_indices = []
seen_classes = set()
for i, s in enumerate(train_dataset.samples):
    c = s["label"]
    if c not in seen_classes:
        seen_classes.add(c)
        mini_indices.append(i)
    if len(seen_classes) == 20:
        break

mini_loader = DataLoader(Subset(train_dataset, mini_indices), batch_size=20, shuffle=False)
sanity_model = ASLGRUClassifier(input_size=143, hidden_size=64, num_layers=1, num_classes=20, dropout=0.0).to(device)
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.0)
optimizer = torch.optim.Adam(sanity_model.parameters(), lr=0.005)

print("Huấn luyện Sanity Check (20 mẫu)... ")
for ep in range(1, 41):
    sanity_model.train()
    for x, y in mini_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = sanity_model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        acc = float((torch.argmax(logits, dim=1) == y).sum().item() / len(y))
    if ep % 10 == 0:
        print(f"  Epoch {ep:2d}/40 -> Loss: {loss.item():.4f} | Accuracy: {acc * 100:.1f}%")

if acc >= 0.95 and loss.item() < 0.15:
    print(" SANITY CHECK ĐẠT: Pipeline học hoàn hảo!")

## 6. Baseline Random Forest (Đặc trưng Thống kê Thời gian)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

test_dataset = ASLLandmarkDataset(splits_dir / "test.csv", landmarks_dir, is_train=False)

def extract_summary(dataset):
    X, y = [], []
    for i in range(len(dataset)):
        x_t, y_t = dataset[i]
        arr = x_t.numpy()
        vec = np.concatenate([np.mean(arr, axis=0), np.std(arr, axis=0), np.min(arr, axis=0), np.max(arr, axis=0)])
        X.append(vec)
        y.append(int(y_t))
    return np.array(X), np.array(y)

X_tr, y_tr = extract_summary(train_dataset)
X_te, y_te = extract_summary(test_dataset)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
rf_preds = rf.predict(X_te)

print(f"Random Forest Test Top-1 Accuracy: {accuracy_score(y_te, rf_preds) * 100:.2f}%")
print(f"Random Forest Test Macro-F1:       {f1_score(y_te, rf_preds, average='macro', zero_division=0):.4f}")

## 7. Huấn Luyện Mô Hình GRU Chính Thức (3 Seeds & Early Stopping)

In [ ]:
from src.models.gru import create_model
from src.training.train import train_gru
from src.training.evaluate import evaluate_test_set

val_dataset = ASLLandmarkDataset(splits_dir / "val.csv", landmarks_dir, is_train=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

checkpoints_dir = repo_root / "models" / "checkpoints"
checkpoints_dir.mkdir(parents=True, exist_ok=True)

seeds = [42, 123, 999]
results = []

for seed in seeds:
    print(f"\n--- Bắt đầu huấn luyện Seed {seed} ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Bật Data Augmentation cho tập train
    train_aug_dataset = ASLLandmarkDataset(splits_dir / "train.csv", landmarks_dir, is_train=True, augment=True)
    train_loader = DataLoader(train_aug_dataset, batch_size=32, shuffle=True)
    
    gru_model = create_model(input_size=143, num_classes=20, hidden_size=128, num_layers=2, dropout=0.3)
    ckpt_file = checkpoints_dir / f"best_gru_seed{seed}.pth"
    
    train_gru(
        model=gru_model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=60,
        lr=0.001,
        weight_decay=1e-4,
        label_smoothing=0.1,
        patience=15,
        device=device,
        checkpoint_path=ckpt_file,
        verbose=True
    )
    
    report = evaluate_test_set(
        model=gru_model,
        test_loader=test_loader,
        test_csv=splits_dir / "test.csv",
        class_names=glosses,
        device=device,
        plots_dir=P.plots,
        report_json_path=P.metrics / f"test_report_seed{seed}.json"
    )
    print(f"Kết quả Test Seed {seed}: Top-1={report['top1_accuracy']*100:.2f}%, Top-3={report['top3_accuracy']*100:.2f}%, F1={report['macro_f1']:.4f}")
    results.append(report)

top1_scores = [r["top1_accuracy"] * 100 for r in results]
top3_scores = [r["top3_accuracy"] * 100 for r in results]
f1_scores = [r["macro_f1"] for r in results]

print("\n" + "=" * 60)
print(f"TỔNG KẾT TEST ACCURACY (3 SEEDS):")
print(f" - Top-1 Accuracy: {np.mean(top1_scores):.2f}% ± {np.std(top1_scores):.2f}%")
print(f" - Top-3 Accuracy: {np.mean(top3_scores):.2f}% ± {np.std(top3_scores):.2f}%")
print(f" - Macro-F1:       {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print("=" * 60)

## 8. Trực Quan Hóa & Đóng Gói Kết Quả

In [ ]:
# Hiển thị Confusion Matrix
cm_img_path = P.plots / "confusion_matrix.png"
if cm_img_path.exists():
    img = plt.imread(str(cm_img_path))
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix (Test Set)", fontsize=14, fontweight="bold")
    plt.show()

# Đóng gói checkpoints và metrics thành file ZIP để tải về máy
bundle_dir = Path("/kaggle/working/asl_outputs") if Path("/kaggle/working").exists() else repo_root / "outputs"
zip_out = shutil.make_archive(str(repo_root / "asl_20gloss_results"), "zip", str(bundle_dir))
print(f" Đã tạo gói kết quả tải về tại: {zip_out}")
print("Tất cả các bước huấn luyện đã hoàn tất xuất sắc!")